# Classification NBA Model

## Libraries

In [ ]:
save_models_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/classifier/"
prefix_name = "classifier_total_bet_"

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from nba_ou.data_processing.missing_data.handle_missing_data import apply_missing_policy, compute_and_save_train_medians

## Load Data

In [ ]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "training_data_from_2004-10-01_to_2026-01-10.csv"
path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [ ]:
df_stats= df_stats[df_stats['TOTAL_POINTS'] > 130]

In [ ]:
df_stats.shape[0]

In [ ]:
df_stats

In [ ]:
#count nas in TOTA_OVER_UNDER_LINE
nans = df_stats['TOTAL_OVER_UNDER_LINE'].isna().sum()
print(f"Number of NaNs in TOTAL_OVER_UNDER_LINE: {nans}")

In [ ]:
df_stats.dropna(subset=['TOTAL_OVER_UNDER_LINE'], inplace=True)
df_stats = df_stats[df_stats['TOTAL_OVER_UNDER_LINE'] > 100]

In [ ]:
df_stats['SEASON_YEAR'].unique()

In [ ]:
#drop season 2007
df_stats = df_stats[df_stats['SEASON_YEAR'] != 2007]

In [ ]:
df_to_train = df_stats.copy()

In [ ]:
text_columns = df_to_train.select_dtypes(include=['object', 'string']).columns

# Print the names of the columns containing text
# print("Text columns:", list(text_columns))
cols_to_drop = ['SEASON_ID', 'GAME_ID', 'SEASON_TYPE',  'TEAM_ID_TEAM_HOME',
       'TEAM_ID_TEAM_AWAY','IS_OVERTIME']

cols_to_drop += [
    "MATCHUP_TEAM_HOME",
    "TEAM_CITY_TEAM_AWAY",
    "TEAM_ABBREVIATION_TEAM_AWAY",
    "TEAM_ABBREVIATION_TEAM_HOME",
    "TEAM_CITY_TEAM_HOME",
    "TEAM_NAME_TEAM_AWAY",
    "MATCHUP_TEAM_AWAY",
    "TEAM_NAME_TEAM_HOME"
]
cols_to_drop += list(text_columns)

cols_to_drop.pop(cols_to_drop.index('GAME_DATE'))

df_to_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')


In [ ]:
#Calculate number fo rows with at least 1 NA in total and in percentage
rows_with_na = df_to_train.isna().any(axis=1).sum()
total_rows = df_to_train.shape[0]
percentage_with_na = (rows_with_na / total_rows) * 100
print(f"Rows with at least one NA: {rows_with_na} out of {total_rows} ({percentage_with_na:.2f}%)")

In [ ]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

In [ ]:
# df_to_train.fillna(0, inplace=True)

In [ ]:
print("Applying missing data policy...")
print(f"Data shape before handling missing data: {df_to_train.shape}")

In [ ]:
    # Training time
train_medians = compute_and_save_train_medians(df_to_train)

# Prediction time
df_to_train, report = apply_missing_policy(
    df_to_train,
    train_medians=train_medians,
    current_total_line_col="TOTAL_OVER_UNDER_LINE",
    drop_mode="strict",)

In [ ]:
print(f"Data shape after handling missing data: {df_to_train.shape}")

In [ ]:
null_rows = df_to_train[df_to_train.isnull().any(axis=1)]
print(f"Total number of null values in the DataFrame: {null_rows.shape[0]}")
print(f"Percentage of null values in the DataFrame: {null_rows.shape[0] / df_to_train.shape[0] * 100:.2f}%")
print(f"Total number of rows in the DataFrame: {df_to_train.shape[0]}")

null_columns = df_to_train.columns[df_to_train.isnull().any()].tolist()
print(f"Columns with null values: {null_columns}")


In [ ]:
df_to_train.drop(columns='GAME_DATE', inplace=True)

In [ ]:
df_to_train['OVER_UNDER'] = np.where(df_to_train['TOTAL_POINTS'] >= df_to_train['TOTAL_OVER_UNDER_LINE'], 1, 0) # 1 if over, 0 if under

In [ ]:
#count 1 and 0 in OVER_UNDER
df_to_train['OVER_UNDER'].value_counts()

## Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df_to_train.drop(cols_to_drop + ['TOTAL_POINTS', 'OVER_UNDER'], axis=1, errors='ignore')
y = df_to_train['OVER_UNDER']

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=16)

In [ ]:
df_to_train['IS_TRAINING_DATA'] = False

# Mark True for the rows in the training set
df_to_train.loc[X_train.index, 'IS_TRAINING_DATA'] = True
output_name = f"{data_path}/training_data_with_missing_data_handled_from_2004-10-01_to_2026-01-10_classifier.csv"
df_to_train.to_csv(output_name, index=False)

In [ ]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

## Cross-validation

In [ ]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, make_scorer, balanced_accuracy_score
)

In [ ]:
# Declare KFold
kf = KFold(n_splits=5, shuffle=True, random_state=16)

In [ ]:
# Declare scores to be used
scoring = {
    'Acc': make_scorer(accuracy_score),
    'Prec': make_scorer(precision_score),
    'Rec': make_scorer(recall_score),
    'Bacc': make_scorer(balanced_accuracy_score),
}

In [ ]:

def print_classification_metrics(cv_results, scoring):
    """
    Prints the mean cross-validation scores for each metric on train and validation folds.
    """
    for sc in scoring.keys():
        print(f"Train {sc}:", cv_results[f'train_{sc}'].mean().round(5))
        print(f"Validation {sc}:", cv_results[f'test_{sc}'].mean().round(5))
        print()

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """
    Plots a basic confusion matrix using Matplotlib.
    """
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    im = ax.imshow(cm, aspect='equal')

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(title)

    # Show text on matrix
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text = ax.text(j, i, cm[i, j],
                           ha="center", va="center", color="w")
    plt.show()
    

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, make_scorer
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## Baseline

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
dummy_clf = DummyClassifier(strategy='most_frequent')
cv_results = cross_validate(
    dummy_clf, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring, 
    return_train_score=True
)
dummy_clf.fit(X_train, y_train)
print("=== Dummy (Most Frequent) Classifier ===")
print_classification_metrics(cv_results, scoring)

In [ ]:
y_train_preds_dummy = dummy_clf.predict(X_train)
plot_confusion_matrix(y_train, y_train_preds_dummy, 
                      title="Dummy Classifier - Confusion Matrix (Train)")

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LogisticRegression(max_iter=1000)
cv_results = cross_validate(
    lr, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring, 
    return_train_score=True,
    n_jobs=-1
)
lr.fit(X_train, y_train)


In [ ]:
print("=== Logistic Regression ===")
print_classification_metrics(cv_results, scoring)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5,
    max_features=0.6,
    random_state=16,
    n_jobs=-1
)

cv_results = cross_validate(
    rf, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring, 
    return_train_score=True, 
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
print("=== Random Forest Classifier ===")
print_classification_metrics(cv_results, scoring)

In [ ]:
y_train_preds_rf = rf.predict(X_train)
plot_confusion_matrix(y_train, y_train_preds_rf,
                      title="Random Forest - Confusion Matrix (Train)")

In [ ]:
xgb_clf = XGBClassifier(
    max_depth=5,
    learning_rate=0.05,
    n_estimators=500,
    subsample=1.0,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=16
)

cv_results = cross_validate(
    xgb_clf, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring, 
    return_train_score=True,
    n_jobs=-1
)

xgb_clf.fit(X_train, y_train)

print("=== XGBClassifier ===")
print_classification_metrics(cv_results, scoring)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """
    Plots a basic confusion matrix using Matplotlib.
    """
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    im = ax.imshow(cm, aspect='equal')

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(title)

    # Show text on matrix
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text = ax.text(j, i, cm[i, j],
                           ha="center", va="center", color="w")
    plt.show()

In [ ]:
y_train_preds_xgb = xgb_clf.predict(X_train)
plot_confusion_matrix(y_train, y_train_preds_xgb, 
                      title="XGB Classifier - Confusion Matrix (Train)")

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

In [ ]:
brf = BalancedRandomForestClassifier(200,
                                     max_depth=None,
                                     sampling_strategy='not minority',
                                     replacement=True,
                                     n_jobs=-1)
brf_cv = cross_validate(brf, X_train, y_train, cv=kf,
                        scoring=scoring, return_train_score=True)

print_classification_metrics(brf_cv, scoring)

## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

### Randomized Search

### Grid Search

In [ ]:
import os
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBClassifier

In [ ]:
def print_rs_clf_scores(gscv_model):
    """
    Prints the best hyperparameters and the corresponding accuracy and errors.
    """
    best_idx = gscv_model.best_index_
    best_params = gscv_model.best_params_
    
    # mean_train_score and mean_test_score hold the average CV train/test accuracies
    train_acc = gscv_model.cv_results_['mean_train_score'][best_idx]
    valid_acc = gscv_model.cv_results_['mean_test_score'][best_idx]
    
    print("Best Parameters:", best_params, "\n")
    print(f"Train Accuracy: {train_acc:.3f}")
    print(f"Validation Accuracy (CV): {valid_acc:.3f}")
    print(f"Train Error: {(1 - train_acc):.3f}")
    print(f"Validation Error (CV): {(1 - valid_acc):.3f}")

In [ ]:
def print_rs_clf_scores(gscv_model):
    """
    Prints the best hyperparameters and the corresponding training and validation metrics.
    """
    best_idx = gscv_model.best_index_
    best_params = gscv_model.best_params_

    # Metrics used during scoring
    metrics = ['Acc', 'Prec', 'Rec', 'Bacc']

    print("Best Parameters:", best_params, "\n")
    
    for metric in metrics:
        train_score = gscv_model.cv_results_.get(f'mean_train_{metric}', None)
        valid_score = gscv_model.cv_results_.get(f'mean_test_{metric}', None)
        
        if train_score is not None and valid_score is not None:
            train_val = train_score[best_idx]
            valid_val = valid_score[best_idx]
            print(f"{metric} - Train: {train_val:.5f}, Validation: {valid_val:.5f}, "
                  f"Train Error: {1 - train_val:.5f}, Validation Error: {1 - valid_val:.5f}")
        else:
            print(f"{metric}: Not available (was not calculated on train set)")

In [ ]:
# Define the base classifier
xgb_clf = XGBClassifier(random_state=16, n_jobs=-1)

# Define hyperparameter grid (similar to your regressor but for classification)
params = [{
    'n_estimators': [300,500],      # # of boosting rounds
    'learning_rate': [0.02, 0.01, 0.03],
    'max_depth': [3,4, 5],
    'min_child_weight': [3, 4,5],      # akin to min_samples_split
    'subsample': [0.8, 0.6,1],
    'colsample_bytree': [0.8, 1],
    'gamma': [0],                    # regularization param
}]

xgb_gs_clf = GridSearchCV(
    estimator=xgb_clf,
    param_grid=params,
    scoring=scoring,
    cv=kf,
    verbose=1,
    refit=lambda x: x["mean_test_Bacc"].argmax(),
    return_train_score=True,
    n_jobs=-1
)

# Fit GridSearchCV
xgb_gs_clf.fit(X_train, y_train) 

In [ ]:
print_rs_clf_scores(xgb_gs_clf)

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label="target",
    problem_type="binary",
    eval_metric="balanced_accuracy",
).fit(
    train_data=X_train.assign(target=y_train),
    time_limit=7000,
    presets="good_quality",
    hyperparameter_tune_kwargs="auto",
    # num_cpus=8,
    hyperparameters={
        "XGB": {}   # only XGBoost models
    }
)

In [ ]:
predictor = TabularPredictor.load("/home/adrian_alvarez/Projects/NBA_over_under_predictor/lab/train_classifier/AutogluonModels/ag-20260123_122303")  # use your actual folder

print("Saved at:", predictor.path)
print("Best model:", predictor.get_model_best())
print("All models:", predictor.get_model_names())

In [ ]:
lb = predictor.leaderboard(silent=True)
lb


In [ ]:
# Keep only the best model (+ its dependencies), delete the rest from disk
predictor.delete_models(models_to_keep="best", delete_from_disk=True)

# (Optional) shrink further by removing auxiliary artifacts not needed for prediction
predictor.save_space(remove_data=True)

# Save back to disk (same folder); after this, the directory is much smaller
predictor.save()